# Stylistic Inspiration Discovery

This notebook helps discover authors and works that would provide stylistic inspiration for fine-tuning, based on a story outline.

## Process

1. **Ingest outline** - Load the story outline/premise
2. **Model strategy discussion** - Conversation to determine different options for how many fine-tuned models to make, and what they could look like
3. **Recommend works** - Suggest works to be inspired from and collect fine-tuning training data from
4. **Confirm selections** - Review and confirm which works to use


In [13]:
### EDIT THESE EVERY NEW STORY ###

import os
import json
import datetime
from openai import OpenAI
from typing import Optional

PROJECT_NAME = "QoGD"

# Model configuration
MODEL_NAME = "gpt-5"

# Setup client
openai_client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# File paths
SOURCE_PROMPTS_DIR = f"source_prompts/{PROJECT_NAME}/"
OUTPUT_JSON = f"{SOURCE_PROMPTS_DIR}selected_inspirations.json"
OUTPUT_CONVERSATION = f"{SOURCE_PROMPTS_DIR}inspiration_conversation.md"

# Ensure directories exist
os.makedirs(SOURCE_PROMPTS_DIR, exist_ok=True)


In [14]:
def load_outline() -> str:
    """Load outline from source_prompts directory."""
    # Try to find outline file - could be write_prompt_1.txt or system_prompt.txt or a dedicated outline file
    possible_files = [
        f"{SOURCE_PROMPTS_DIR}outline.txt",
        f"{SOURCE_PROMPTS_DIR}write_prompt_1.txt",
        f"{SOURCE_PROMPTS_DIR}system_prompt.txt",
    ]
    
    for file_path in possible_files:
        if os.path.exists(file_path):
            with open(file_path, 'r') as f:
                return f.read()
    
    return None

def make_llm_call(messages, model=None):
    """Make LLM call with OpenAI client."""
    model = model or MODEL_NAME
    
    response = openai_client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# Load outline
outline = load_outline()
if outline:
    print(f"Loaded outline ({len(outline)} characters)")
    print("\n" + "="*80)
    print(outline[:500] + "..." if len(outline) > 500 else outline)
    print("="*80)
else:
    print("Warning: No outline file found. You can provide it manually in the next cell.")


Loaded outline (3920 characters)

Write the opening scene (part 1). Here is the outline.

### Beliefs explored

1. There is no simulation - we already exited it. We're outside of Plato's cave. We can go back in - it's warm and cozy inside. Womb, cocoon, and death.
2. There's a particular set of roles that women ought to play in society. Posits this is both true and false (meaningless) by presenting a depiction where both sides of the argument could say, "see - the story aligns with my beliefs".
3. The way we come to understand s...


In [15]:
# Optional: Manually set outline if not found or to override
# outline = """Your outline text here"""


In [16]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Javascript
import json
import time
import threading

class StylisticInspirationUI:
    """Interactive UI for discovering stylistic inspiration sources."""
    
    def __init__(self, outline_text: str):
        self.outline = outline_text
        self.selected_works = []
        self.model_strategy = None
        self.current_stage = 0  # 0=start, 1=strategy, 2=recommendations, 3=confirmation
        
        # System prompts for different stages
        self.strategy_system_prompt = """You are a literary and AI model advisor helping determine the fine-tuning strategy for a story.
Based on a story outline, help determine:
1. How many fine-tuned models should be created
2. What each model should specialize in (e.g., different narrative voices, stylistic approaches, character perspectives)
3. Why this approach makes sense for this particular story

Consider factors like:
- Multiple perspectives or narrative voices in the story
- Different thematic sections that might benefit from different styles
- Whether a single unified model or multiple specialized models would better serve the story
- The complexity and variety of the story's stylistic needs

Provide clear, well-reasoned recommendations with explanations."""

        self.recommendations_system_prompt = """You are a literary advisor specializing in helping writers find stylistic inspiration for their work.
Based on a story outline and a decided fine-tuning strategy (how many models and what each should do), suggest authors and specific works that would provide stylistic inspiration for training data.

For each model in the strategy, suggest relevant works that would help capture the required style, themes, or techniques.

When suggesting works, provide:
1. Author name
2. Specific work title
3. Which model(s) this work relates to
4. Brief explanation of why this work is relevant (style, themes, techniques, etc.)

Format your suggestions clearly with author and title on separate lines or clearly marked."""

        # Widgets
        self.stage_display = widgets.HTML(
            value="<p><strong>Current Stage:</strong> Ready to begin</p>",
            layout=widgets.Layout(width="100%")
        )
        
        self.strategy_display = widgets.HTML(
            value="<p><em>Model strategy will appear here after discussion.</em></p>",
            layout=widgets.Layout(width="100%", height="200px", overflow="auto")
        )
        
        self.suggestions_display = widgets.HTML(
            value="<p><em>Work recommendations will appear here.</em></p>",
            layout=widgets.Layout(width="100%", height="300px", overflow="auto")
        )
        
        # Selection widgets for recommendations
        self.work_selection_widgets = widgets.VBox([])
        self.select_all_button = widgets.Button(
            description="Select All",
            button_style="info",
            layout=widgets.Layout(width="150px")
        )
        self.deselect_all_button = widgets.Button(
            description="Deselect All",
            button_style="",
            layout=widgets.Layout(width="150px")
        )
        self.add_selected_button = widgets.Button(
            description="Add Selected Works",
            button_style="success",
            layout=widgets.Layout(width="200px")
        )
        self.work_checkboxes = {}  # Store checkboxes by work ID
        
        # Step 2: Strategy Discussion Chat UI
        self.strategy_chat_display = widgets.HTML(
            value="",
            layout=widgets.Layout(width="100%", height="300px", overflow="auto")
        )
        self.strategy_user_input = widgets.Textarea(
            value="",
            description="Your message:",
            layout=widgets.Layout(width="100%", height="80px"),
            placeholder="Ask questions about the model strategy..."
        )
        self.strategy_ask_button = widgets.Button(
            description="Ask Follow-up",
            button_style="info",
            layout=widgets.Layout(width="200px")
        )
        
        # Step 3: Work Recommendations Chat UI
        self.recommendations_chat_display = widgets.HTML(
            value="",
            layout=widgets.Layout(width="100%", height="300px", overflow="auto")
        )
        self.recommendations_user_input = widgets.Textarea(
            value="",
            description="Your message:",
            layout=widgets.Layout(width="100%", height="80px"),
            placeholder="Ask questions about work recommendations..."
        )
        self.recommendations_ask_button = widgets.Button(
            description="Ask Follow-up",
            button_style="info",
            layout=widgets.Layout(width="200px")
        )
        
        # Conversation history for each step
        self.strategy_conversation_history = []
        self.recommendations_conversation_history = []
        
        self.selected_works_display = widgets.HTML(
            value="<p><strong>Selected works:</strong> None</p>",
            layout=widgets.Layout(width="100%", min_height="150px")
        )
        
        # Loading indicator
        self.loading_indicator = widgets.HTML(
            value="",
            layout=widgets.Layout(width="100%", padding="20px", display="none")
        )
        
        # Confirmation summary panel (will be initialized after method is defined)
        self.confirmation_summary = widgets.HTML(
            value="<p>Loading...</p>",
            layout=widgets.Layout(width="100%", padding="15px", border="2px solid #ddd", background="#f9f9f9")
        )
        
        # Buttons
        self.start_strategy_button = widgets.Button(
            description="Start Strategy Discussion",
            button_style="primary",
            layout=widgets.Layout(width="200px")
        )
        
        self.confirm_strategy_button = widgets.Button(
            description="Confirm Strategy",
            button_style="success",
            layout=widgets.Layout(width="200px")
        )
        
        self.get_recommendations_button = widgets.Button(
            description="Get Work Recommendations",
            button_style="primary",
            layout=widgets.Layout(width="200px")
        )
        
        self.add_selection_button = widgets.Button(
            description="Add to Selection",
            button_style="success",
            layout=widgets.Layout(width="200px")
        )
        
        self.finalize_button = widgets.Button(
            description="Finalize & Save",
            button_style="warning",
            layout=widgets.Layout(width="200px")
        )
        
        self.output = widgets.Output()
        
        # Event handlers
        self.start_strategy_button.on_click(self.start_strategy_discussion)
        self.confirm_strategy_button.on_click(self.confirm_strategy)
        self.get_recommendations_button.on_click(self.get_work_recommendations)
        self.strategy_ask_button.on_click(self.ask_strategy_followup)
        self.recommendations_ask_button.on_click(self.ask_recommendations_followup)
        self.add_selection_button.on_click(self.add_to_selection)
        self.add_selected_button.on_click(self.add_selected_works)
        self.select_all_button.on_click(self.select_all_works)
        self.deselect_all_button.on_click(self.deselect_all_works)
        self.finalize_button.on_click(self.save_selections)
        
        # Current suggestions (parsed from last LLM response)
        self.current_suggestions_text = ""
        self.current_strategy_text = ""
        self.parsed_recommendations = []  # List of parsed work recommendations
        
        # Initialize displays (methods must be defined first)
        self._update_stage_display()
        self._update_confirmation_summary()
        
    def display(self):
        """Display the UI."""
        display(widgets.VBox([
            widgets.HTML("<h3>Stylistic Inspiration Discovery</h3>"),
            self.stage_display,
            self.confirmation_summary,
            self.loading_indicator,
            widgets.HBox([
                widgets.VBox([
                    widgets.HTML("<h4>Step 2: Model Strategy Discussion</h4>"),
                    self.strategy_display,
                    widgets.HBox([
                        self.start_strategy_button,
                        self.confirm_strategy_button,
                    ]),
                    widgets.HTML("<h5>Strategy Discussion:</h5>"),
                    self.strategy_chat_display,
                    self.strategy_user_input,
                    self.strategy_ask_button,
                    widgets.HTML("<hr><h4>Step 3: Work Recommendations</h4>"),
                    self.suggestions_display,
                    widgets.HBox([
                        self.get_recommendations_button,
                        self.add_selection_button,
                    ]),
                    widgets.HTML("<h5>Select Works from Recommendations:</h5>"),
                    widgets.HBox([
                        self.select_all_button,
                        self.deselect_all_button,
                        self.add_selected_button,
                    ]),
                    self.work_selection_widgets,
                    widgets.HTML("<h5>Recommendations Discussion:</h5>"),
                    self.recommendations_chat_display,
                    self.recommendations_user_input,
                    self.recommendations_ask_button,
                ], layout=widgets.Layout(width="70%")),
                widgets.VBox([
                    widgets.HTML("<h4>Step 4: Selected Works</h4>"),
                    self.selected_works_display,
                    self.finalize_button,
                ], layout=widgets.Layout(width="30%")),
            ]),
            self.output
        ]))
    
    def _show_loading(self, message="Calling LLM..."):
        """Show loading indicator and disable buttons."""
        spinner_html = f"""
        <div style="background: #f0f0f0; border: 2px solid #2196F3; border-radius: 5px; padding: 20px; text-align: center;">
            <div style="display: inline-block; width: 40px; height: 40px; border: 4px solid #f3f3f3; border-top: 4px solid #2196F3; border-radius: 50%; animation: spin 1s linear infinite; margin-right: 15px;"></div>
            <span style="font-size: 16px; font-weight: bold; color: #2196F3; vertical-align: middle;">{message}</span>
        </div>
        <style>
            @keyframes spin {{
                0% {{ transform: rotate(0deg); }}
                100% {{ transform: rotate(360deg); }}
            }}
        </style>
        """
        self.loading_indicator.value = spinner_html
        self.loading_indicator.layout.display = "block"
        
        # Disable all buttons
        self.start_strategy_button.disabled = True
        self.confirm_strategy_button.disabled = True
        self.get_recommendations_button.disabled = True
        self.strategy_ask_button.disabled = True
        self.recommendations_ask_button.disabled = True
        self.add_selection_button.disabled = True
        self.add_selected_button.disabled = True
        self.select_all_button.disabled = True
        self.deselect_all_button.disabled = True
        self.finalize_button.disabled = True
    
    def _hide_loading(self):
        """Hide loading indicator and enable buttons."""
        self.loading_indicator.value = ""
        self.loading_indicator.layout.display = "none"
        
        # Re-enable buttons
        self.start_strategy_button.disabled = False
        self.confirm_strategy_button.disabled = False
        self.get_recommendations_button.disabled = False
        self.strategy_ask_button.disabled = False
        self.recommendations_ask_button.disabled = False
        self.add_selection_button.disabled = False
        self.add_selected_button.disabled = False
        self.select_all_button.disabled = False
        self.deselect_all_button.disabled = False
        self.finalize_button.disabled = False
    
    def _generate_confirmation_summary(self):
        """Generate HTML summary of what has been confirmed."""
        html = "<div style='font-family: Arial, sans-serif;'>"
        html += "<h4 style='margin-top: 0; color: #333; border-bottom: 2px solid #2196F3; padding-bottom: 10px;'>📋 Confirmation Summary</h4>"
        
        # Step 1: Outline (always present)
        html += "<div style='margin: 10px 0; padding: 10px; background: #e8f5e9; border-left: 4px solid #4CAF50; border-radius: 4px;'>"
        html += "<strong style='color: #2e7d32;'>✓ Step 1: Outline</strong> - Loaded"
        html += "</div>"
        
        # Step 2: Model Strategy
        if self.model_strategy:
            html += "<div style='margin: 10px 0; padding: 10px; background: #e8f5e9; border-left: 4px solid #4CAF50; border-radius: 4px;'>"
            html += "<strong style='color: #2e7d32;'>✓ Step 2: Model Strategy - CONFIRMED</strong><br>"
            # Show preview of strategy (first 200 chars)
            strategy_preview = self.model_strategy[:200] + "..." if len(self.model_strategy) > 200 else self.model_strategy
            # Escape HTML and convert newlines to <br>
            import html as html_module
            strategy_preview_escaped = html_module.escape(strategy_preview).replace('\n', '<br>')
            html += f"<div style='margin-top: 8px; padding: 8px; background: white; border-radius: 3px; font-size: 0.9em; color: #555;'>{strategy_preview_escaped}</div>"
            html += "</div>"
        elif self.current_strategy_text:
            html += "<div style='margin: 10px 0; padding: 10px; background: #fff3e0; border-left: 4px solid #FF9800; border-radius: 4px;'>"
            html += "<strong style='color: #E65100;'>⚠ Step 2: Model Strategy - PENDING CONFIRMATION</strong><br>"
            html += "<em style='font-size: 0.9em; color: #666;'>Strategy has been generated but not yet confirmed. Click 'Confirm Strategy' to proceed.</em>"
            html += "</div>"
        else:
            html += "<div style='margin: 10px 0; padding: 10px; background: #f5f5f5; border-left: 4px solid #9e9e9e; border-radius: 4px;'>"
            html += "<strong style='color: #616161;'>○ Step 2: Model Strategy</strong> - Not started"
            html += "</div>"
        
        # Step 3: Work Recommendations
        if self.selected_works:
            html += "<div style='margin: 10px 0; padding: 10px; background: #e8f5e9; border-left: 4px solid #4CAF50; border-radius: 4px;'>"
            html += f"<strong style='color: #2e7d32;'>✓ Step 3: Selected Works - {len(self.selected_works)} work(s) selected</strong><br>"
            html += "<div style='margin-top: 8px; padding: 8px; background: white; border-radius: 3px; font-size: 0.9em;'>"
            html += "<ul style='margin: 0; padding-left: 20px;'>"
            for work in self.selected_works:
                model_info = f" <em>(for {work.get('model', 'all models')})</em>" if work.get('model') else ""
                html += f"<li style='margin: 5px 0;'><strong>{work['author']}</strong> - {work['title']}{model_info}</li>"
            html += "</ul>"
            html += "</div>"
            html += "</div>"
        elif self.current_suggestions_text:
            html += "<div style='margin: 10px 0; padding: 10px; background: #fff3e0; border-left: 4px solid #FF9800; border-radius: 4px;'>"
            html += "<strong style='color: #E65100;'>⚠ Step 3: Work Recommendations - PENDING SELECTION</strong><br>"
            html += "<em style='font-size: 0.9em; color: #666;'>Recommendations have been generated. Add works to your selection to proceed.</em>"
            html += "</div>"
        else:
            html += "<div style='margin: 10px 0; padding: 10px; background: #f5f5f5; border-left: 4px solid #9e9e9e; border-radius: 4px;'>"
            html += "<strong style='color: #616161;'>○ Step 3: Work Recommendations</strong> - Not started"
            html += "</div>"
        
        # Step 4: Final confirmation
        if self.model_strategy and self.selected_works:
            html += "<div style='margin: 10px 0; padding: 10px; background: #e3f2fd; border-left: 4px solid #2196F3; border-radius: 4px;'>"
            html += "<strong style='color: #1565C0;'>→ Step 4: Ready to Finalize</strong><br>"
            html += "<em style='font-size: 0.9em; color: #666;'>All steps complete. Click 'Finalize & Save' to save your selections.</em>"
            html += "</div>"
        else:
            html += "<div style='margin: 10px 0; padding: 10px; background: #f5f5f5; border-left: 4px solid #9e9e9e; border-radius: 4px;'>"
            html += "<strong style='color: #616161;'>○ Step 4: Finalize & Save</strong> - Waiting for previous steps"
            html += "</div>"
        
        html += "</div>"
        return html
    
    def _update_confirmation_summary(self):
        """Update the confirmation summary display."""
        self.confirmation_summary.value = self._generate_confirmation_summary()
    
    def _update_stage_display(self):
        """Update the stage indicator."""
        stages = [
            "Ready to begin",
            "Discussing model strategy",
            "Getting work recommendations",
            "Confirming selections"
        ]
        stage_colors = ["#666", "#2196F3", "#4CAF50", "#FF9800"]
        self.stage_display.value = f"<p style='padding: 10px; background: {stage_colors[self.current_stage]}; color: white; border-radius: 5px;'><strong>Current Stage:</strong> {stages[self.current_stage]}</p>"
        
    def start_strategy_discussion(self, _):
        """Start the model strategy discussion."""
        self.current_stage = 1
        self._update_stage_display()
        
        self._show_loading("Generating model strategy suggestions...")
        
        with self.output:
            clear_output()
        
        initial_prompt = f"""Based on this story outline, help me determine the fine-tuning strategy:

Outline:
{self.outline}

Please suggest:
1. How many fine-tuned models should be created for this story
2. What each model should specialize in (e.g., different narrative voices, stylistic approaches, character perspectives)
3. Why this approach makes sense for this particular story

Provide 2-3 different strategy options if there are multiple viable approaches."""

        messages = [
            {"role": "system", "content": self.strategy_system_prompt},
            {"role": "user", "content": initial_prompt}
        ]
        
        try:
            response = make_llm_call(messages)
            self.current_strategy_text = response
            
            # Update conversation history
            self.strategy_conversation_history.append({"role": "user", "content": initial_prompt})
            self.strategy_conversation_history.append({"role": "assistant", "content": response})
            
            # Update displays
            self.strategy_display.value = f"<div style='white-space: pre-wrap;'>{response}</div>"
            self._update_strategy_chat_display()
            self._update_confirmation_summary()
            
            self._hide_loading()
            
            with self.output:
                clear_output()
                print("✓ Got strategy suggestions! Review and discuss, then click 'Confirm Strategy' when ready.")
                
        except Exception as e:
            self._hide_loading()
            with self.output:
                clear_output()
                print(f"Error: {e}")
    
    def confirm_strategy(self, _):
        """Confirm the model strategy and move to recommendations stage."""
        if not self.current_strategy_text:
            with self.output:
                clear_output()
                print("Please start the strategy discussion first.")
            return
        
        # Show confirmation summary before confirming
        with self.output:
            clear_output()
            print("=" * 80)
            print("CONFIRMING MODEL STRATEGY")
            print("=" * 80)
            print("\nYou are about to confirm the following strategy:\n")
            print("-" * 80)
            print(self.current_strategy_text)
            print("-" * 80)
            print("\n✓ This strategy has been confirmed and locked in.")
            print("You can now proceed to Step 3: Get Work Recommendations")
            print("=" * 80)
        
        # Store the confirmed strategy
        self.model_strategy = self.current_strategy_text
        self.current_stage = 2
        self._update_stage_display()
        self._update_confirmation_summary()
    
    def get_work_recommendations(self, _):
        """Get work recommendations based on the confirmed strategy."""
        if not self.model_strategy:
            with self.output:
                clear_output()
                print("Please confirm the model strategy first (Step 2).")
            return
        
        self.current_stage = 2
        self._update_stage_display()
        
        self._show_loading("Finding inspirational works for training data...")
        
        with self.output:
            clear_output()
        
        recommendations_prompt = f"""Based on this story outline and the confirmed fine-tuning strategy, suggest authors and specific works that would provide stylistic inspiration for training data.

Story Outline:
{self.outline}

Confirmed Model Strategy:
{self.model_strategy}

Please suggest works for each model in the strategy. For each suggestion, provide:
- Author name
- Specific work title
- Which model(s) this work relates to
- Why this work is relevant (style, themes, techniques, etc.)

Format your response clearly."""

        messages = [
            {"role": "system", "content": self.recommendations_system_prompt},
            {"role": "user", "content": recommendations_prompt}
        ]
        
        try:
            response = make_llm_call(messages)
            self.current_suggestions_text = response
            
            # Update conversation history
            self.recommendations_conversation_history.append({"role": "user", "content": recommendations_prompt})
            self.recommendations_conversation_history.append({"role": "assistant", "content": response})
            
            # Parse recommendations and create selectable interface
            self._parse_recommendations(response)
            
            # Update displays
            self.suggestions_display.value = f"<div style='white-space: pre-wrap;'>{response}</div>"
            self._update_recommendations_chat_display()
            self._update_confirmation_summary()
            
            self._hide_loading()
            
            with self.output:
                clear_output()
                print("✓ Got work recommendations! Select works using checkboxes below.")
                
        except Exception as e:
            self._hide_loading()
            with self.output:
                clear_output()
                print(f"Error: {e}")
    
    def ask_strategy_followup(self, _):
        """Ask a follow-up question about the strategy."""
        user_question = self.strategy_user_input.value.strip()
        if not user_question:
            with self.output:
                clear_output()
                print("Please enter a question first.")
            return
        
        self._show_loading("Getting response...")
        
        with self.output:
            clear_output()
        
        messages = [
            {"role": "system", "content": self.strategy_system_prompt},
            {"role": "user", "content": f"Here is the outline:\n\n{self.outline}"}
        ]
        
        # Add conversation history
        messages.extend(self.strategy_conversation_history[-6:])  # Last 3 exchanges
        
        # Add current question
        messages.append({"role": "user", "content": user_question})
        
        try:
            response = make_llm_call(messages)
            
            # Update conversation history
            self.strategy_conversation_history.append({"role": "user", "content": user_question})
            self.strategy_conversation_history.append({"role": "assistant", "content": response})
            
            # Update displays
            self.current_strategy_text = response
            self.strategy_display.value = f"<div style='white-space: pre-wrap;'>{response}</div>"
            self._update_strategy_chat_display()
            self._update_confirmation_summary()
            self.strategy_user_input.value = ""  # Clear input
            
            self._hide_loading()
            
            with self.output:
                clear_output()
                print("✓ Got response!")
                
        except Exception as e:
            self._hide_loading()
            with self.output:
                clear_output()
                print(f"Error: {e}")
    
    def ask_recommendations_followup(self, _):
        """Ask a follow-up question about the recommendations."""
        user_question = self.recommendations_user_input.value.strip()
        if not user_question:
            with self.output:
                clear_output()
                print("Please enter a question first.")
            return
        
        self._show_loading("Getting response...")
        
        with self.output:
            clear_output()
        
        messages = [
            {"role": "system", "content": self.recommendations_system_prompt},
            {"role": "user", "content": f"Here is the outline:\n\n{self.outline}\n\nModel Strategy:\n{self.model_strategy or 'Not yet confirmed'}"}
        ]
        
        # Add conversation history
        messages.extend(self.recommendations_conversation_history[-6:])  # Last 3 exchanges
        
        # Add current question
        messages.append({"role": "user", "content": user_question})
        
        try:
            response = make_llm_call(messages)
            
            # Update conversation history
            self.recommendations_conversation_history.append({"role": "user", "content": user_question})
            self.recommendations_conversation_history.append({"role": "assistant", "content": response})
            
            # Update displays
            self.current_suggestions_text = response
            self.suggestions_display.value = f"<div style='white-space: pre-wrap;'>{response}</div>"
            self._update_recommendations_chat_display()
            
            # Re-parse if recommendations changed
            if "author" in response.lower() and "title" in response.lower():
                self._parse_recommendations(response)
            
            self._update_confirmation_summary()
            self.recommendations_user_input.value = ""  # Clear input
            
            self._hide_loading()
            
            with self.output:
                clear_output()
                print("✓ Got response!")
                
        except Exception as e:
            self._hide_loading()
            with self.output:
                clear_output()
                print(f"Error: {e}")
    
    def add_to_selection(self, _):
        """Add current suggestion to selection (manual parsing)."""
        if not self.current_suggestions_text:
            with self.output:
                clear_output()
                print("No suggestions to add. Get recommendations first.")
            return
        
        # Ask user to specify which work(s) to add
        with self.output:
            clear_output()
            print("Current recommendations:")
            print("-" * 80)
            print(self.current_suggestions_text)
            print("-" * 80)
            print("\nPlease manually specify the author and title to add.")
            print("You can copy from the recommendations above.")
        
        # Create input widgets for manual entry
        author_input = widgets.Text(description="Author:", layout=widgets.Layout(width="400px"))
        title_input = widgets.Text(description="Title:", layout=widgets.Layout(width="400px"))
        model_input = widgets.Text(description="For Model(s):", layout=widgets.Layout(width="400px"), placeholder="e.g., Model 1, or leave blank if for all")
        reason_input = widgets.Textarea(description="Reason:", layout=widgets.Layout(width="400px", height="100px"))
        confirm_button = widgets.Button(description="Add", button_style="success")
        cancel_button = widgets.Button(description="Cancel", button_style="")
        
        def confirm_add(_):
            author = author_input.value.strip()
            title = title_input.value.strip()
            model = model_input.value.strip()
            reason = reason_input.value.strip() or "Added from recommendations"
            
            if author and title:
                work = {
                    "author": author,
                    "title": title,
                    "model": model if model else None,
                    "reason": reason,
                    "added_date": datetime.datetime.now().isoformat()
                }
                self.selected_works.append(work)
                self._update_selected_display()
                self._update_confirmation_summary()
                with self.output:
                    clear_output()
                    print(f"✓ Added: {author} - {title}")
                    print(f"Total works selected: {len(self.selected_works)}")
                # Hide the input widgets
                input_box.close()
            else:
                with self.output:
                    clear_output()
                    print("Please provide both author and title.")
        
        def cancel_add(_):
            input_box.close()
            with self.output:
                clear_output()
        
        confirm_button.on_click(confirm_add)
        cancel_button.on_click(cancel_add)
        
        input_box = widgets.VBox([
            widgets.HTML("<strong>Add Work to Selection:</strong>"),
            author_input,
            title_input,
            model_input,
            reason_input,
            widgets.HBox([confirm_button, cancel_button])
        ])
        
        display(input_box)
    
    def _update_strategy_chat_display(self):
        """Update strategy conversation display."""
        html = "<div style='max-height: 300px; overflow-y: auto;'>"
        for msg in self.strategy_conversation_history:
            role = msg["role"]
            import html as html_module
            content = html_module.escape(msg["content"]).replace("\n", "<br>")
            if role == "user":
                html += f"<p><strong>You:</strong><br>{content}</p>"
            else:
                html += f"<p><strong>Assistant:</strong><br>{content}</p>"
            html += "<hr>"
        html += "</div>"
        self.strategy_chat_display.value = html
    
    def _update_recommendations_chat_display(self):
        """Update recommendations conversation display."""
        html = "<div style='max-height: 300px; overflow-y: auto;'>"
        for msg in self.recommendations_conversation_history:
            role = msg["role"]
            import html as html_module
            content = html_module.escape(msg["content"]).replace("\n", "<br>")
            if role == "user":
                html += f"<p><strong>You:</strong><br>{content}</p>"
            else:
                html += f"<p><strong>Assistant:</strong><br>{content}</p>"
            html += "<hr>"
        html += "</div>"
        self.recommendations_chat_display.value = html
    
    def _parse_recommendations(self, recommendations_text):
        """Parse recommendations text to extract structured work information."""
        # Use LLM to parse recommendations into structured format
        parse_prompt = f"""Extract all book/work recommendations from this text and return them as a JSON array.
For each work, extract:
- author: the author's name
- title: the work's title
- model: which model(s) this relates to (if mentioned)
- reason: why this work is relevant

Text to parse:
{recommendations_text}

Return ONLY a valid JSON array, no other text. Example format:
[
  {{"author": "Author Name", "title": "Book Title", "model": "Model 1", "reason": "Why it's relevant"}},
  ...
]"""

        messages = [
            {"role": "system", "content": "You are a JSON parser. Extract structured data and return ONLY valid JSON."},
            {"role": "user", "content": parse_prompt}
        ]
        
        try:
            response = make_llm_call(messages, model="gpt-4o-mini")  # Use cheaper model for parsing
            # Clean response - remove markdown code blocks if present
            if "```json" in response:
                response = response.split("```json")[1].split("```")[0]
            elif "```" in response:
                response = response.split("```")[1].split("```")[0]
            
            import json
            self.parsed_recommendations = json.loads(response.strip())
            self._create_work_selection_widgets()
        except Exception as e:
            # If parsing fails, create empty selection interface
            self.parsed_recommendations = []
            with self.output:
                print(f"Note: Could not auto-parse recommendations ({e}). You can manually add works.")
    
    def _create_work_selection_widgets(self):
        """Create checkbox widgets for selecting works."""
        self.work_checkboxes = {}
        checkbox_widgets = []
        
        for i, work in enumerate(self.parsed_recommendations):
            work_id = f"work_{i}"
            checkbox = widgets.Checkbox(
                value=False,
                description=f"{work.get('author', 'Unknown')} - {work.get('title', 'Unknown')}",
                layout=widgets.Layout(width="100%", margin="5px 0")
            )
            self.work_checkboxes[work_id] = {
                "checkbox": checkbox,
                "work": work
            }
            
            # Create info display
            model_info = f"<em>(for {work.get('model', 'all models')})</em>" if work.get('model') else ""
            reason = work.get('reason', '')[:100] + "..." if len(work.get('reason', '')) > 100 else work.get('reason', '')
            info_html = widgets.HTML(
                value=f"<div style='margin-left: 30px; font-size: 0.9em; color: #666;'>{model_info} {reason}</div>",
                layout=widgets.Layout(width="100%")
            )
            
            checkbox_widgets.append(widgets.VBox([checkbox, info_html]))
        
        if checkbox_widgets:
            self.work_selection_widgets.children = checkbox_widgets
        else:
            self.work_selection_widgets.children = [widgets.HTML(value="<p><em>No recommendations parsed. Try manually adding works.</em></p>")]
    
    def select_all_works(self, _):
        """Select all work checkboxes."""
        for work_data in self.work_checkboxes.values():
            work_data["checkbox"].value = True
    
    def deselect_all_works(self, _):
        """Deselect all work checkboxes."""
        for work_data in self.work_checkboxes.values():
            work_data["checkbox"].value = False
    
    def add_selected_works(self, _):
        """Add all selected works to the selection."""
        added_count = 0
        for work_id, work_data in self.work_checkboxes.items():
            if work_data["checkbox"].value:
                work = work_data["work"]
                # Check if already added
                already_added = any(
                    w.get("author") == work.get("author") and w.get("title") == work.get("title")
                    for w in self.selected_works
                )
                if not already_added:
                    selected_work = {
                        "author": work.get("author", "Unknown"),
                        "title": work.get("title", "Unknown"),
                        "model": work.get("model"),
                        "reason": work.get("reason", "Added from recommendations"),
                        "added_date": datetime.datetime.now().isoformat()
                    }
                    self.selected_works.append(selected_work)
                    added_count += 1
        
        if added_count > 0:
            self._update_selected_display()
            self._update_confirmation_summary()
            with self.output:
                clear_output()
                print(f"✓ Added {added_count} work(s) to selection!")
        else:
            with self.output:
                clear_output()
                print("No works selected or all selected works were already added.")
    
    def _update_selected_display(self):
        """Update selected works display."""
        if not self.selected_works:
            self.selected_works_display.value = "<p><strong>Selected works:</strong> None</p>"
            return
        
        html = "<div><strong>Selected works:</strong><ul>"
        for i, work in enumerate(self.selected_works):
            html += f"<li><strong>{work['author']}</strong> - {work['title']}"
            if work.get('model'):
                html += f"<br><small><em>For: {work['model']}</em></small>"
            html += f"<br><small>{work['reason']}</small>"
            html += f"<br><button onclick='javascript:void(0)' style='margin-top: 5px;' id='remove_{i}'>Remove</button></li>"
        html += "</ul></div>"
        self.selected_works_display.value = html
    
    def save_selections(self, _):
        """Finalize and save all selections including model strategy."""
        if not self.selected_works:
            with self.output:
                clear_output()
                print("No works selected. Add some works first.")
            return
        
        if not self.model_strategy:
            with self.output:
                clear_output()
                print("Model strategy not confirmed. Please confirm the strategy first (Step 2).")
            return
        
        # Show final confirmation summary
        with self.output:
            clear_output()
            print("=" * 80)
            print("FINALIZING AND SAVING")
            print("=" * 80)
            print(f"\nModel Strategy ({len(self.model_strategy)} characters):")
            print("-" * 80)
            print(self.model_strategy[:300] + "..." if len(self.model_strategy) > 300 else self.model_strategy)
            print("-" * 80)
            print(f"\nSelected Works ({len(self.selected_works)} total):")
            for i, work in enumerate(self.selected_works, 1):
                model_info = f" (for {work.get('model', 'all models')})" if work.get('model') else ""
                print(f"  {i}. {work['author']} - {work['title']}{model_info}")
            print("-" * 80)
            print("\nSaving to files...")
        
        self.current_stage = 3
        self._update_stage_display()
        self._update_confirmation_summary()
        
        # Prepare data to save
        output_data = {
            "project_name": PROJECT_NAME,
            "created_date": datetime.datetime.now().isoformat(),
            "model_strategy": self.model_strategy,
            "outline": self.outline,
            "selected_works": self.selected_works
        }
        
        # Save JSON
        with open(OUTPUT_JSON, 'w') as f:
            json.dump(output_data, f, indent=2)
        
        # Save conversation history
        conversation_md = f"# Inspiration Discovery Conversation\n\n"
        conversation_md += f"Date: {datetime.datetime.now().isoformat()}\n\n"
        conversation_md += f"## Project\n\n{PROJECT_NAME}\n\n"
        conversation_md += f"## Outline\n\n```\n{self.outline}\n```\n\n"
        conversation_md += f"## Model Strategy\n\n{self.model_strategy}\n\n"
        
        # Strategy Discussion Conversation
        conversation_md += f"## Step 2: Strategy Discussion Conversation\n\n"
        for msg in self.strategy_conversation_history:
            role = msg["role"].upper()
            content = msg["content"]
            conversation_md += f"### {role}\n\n{content}\n\n---\n\n"
        
        # Recommendations Discussion Conversation
        conversation_md += f"## Step 3: Recommendations Discussion Conversation\n\n"
        for msg in self.recommendations_conversation_history:
            role = msg["role"].upper()
            content = msg["content"]
            conversation_md += f"### {role}\n\n{content}\n\n---\n\n"
        
        conversation_md += f"## Selected Works\n\n"
        for work in self.selected_works:
            conversation_md += f"- **{work['author']}** - {work['title']}\n"
            if work.get('model'):
                conversation_md += f"  - For Model(s): {work['model']}\n"
            conversation_md += f"  - Reason: {work['reason']}\n"
            conversation_md += f"  - Added: {work['added_date']}\n\n"
        
        with open(OUTPUT_CONVERSATION, 'w') as f:
            f.write(conversation_md)
        
        with self.output:
            clear_output()
            print(f"✓ Finalized and saved!")
            print(f"✓ Model strategy saved")
            print(f"✓ {len(self.selected_works)} works saved to {OUTPUT_JSON}")
            print(f"✓ Full conversation saved to {OUTPUT_CONVERSATION}")

# Initialize UI
if outline:
    ui = StylisticInspirationUI(outline)
    ui.display()
else:
    print("Please set the outline variable first (in the cell above).")
